In [7]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_subjects, derive_events_subject

cfg = load_config('../configs/eye_eeg_simul.yaml')
subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj2', 'subj20', 'subj28', 'subj42', 'subj7']
Subjects: 30


In [8]:
# Test with the first subject
test_subject = subjects[0]
print(f"\nDeriving events for {test_subject}\n")

success = derive_events_subject(cfg, test_subject, overwrite=True, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")


Deriving events for subj3

[subj3] deriving events
   raw events: 1680 total, 31 unique codes
   rule 'tag_memory_array_by_cue_type':
      20 -> 1020: 120 event(s)
      20 -> 1120: 120 event(s)
   derived events: 1920 total (+240 vs raw)
   saved: subj3_derived_events_eve.fif
   saved: subj3_event_recoding_log.txt

Result: OK


In [10]:
import mne
import numpy as np
from eeg_toolkit import get_subject_path
from eeg_toolkit.event_codes import _get_derived_events_path

# Compare raw vs derived
events_raw = mne.read_events(get_subject_path(cfg, test_subject, 'preprocessed_events'))
events_derived = mne.read_events(_get_derived_events_path(cfg, test_subject))

print(f"Raw events:     {len(events_raw)}")
print(f"Derived events: {len(events_derived)}")
print(f"Difference:     +{len(events_derived) - len(events_raw)} new events\n")

# Show distribution of codes
print("=== Derived event codes (only new ones) ===")
unique, counts = np.unique(events_derived[:, 2], return_counts=True)
for c, n in zip(unique, counts):
    if c >= 1000:  # new derived codes
        print(f"  {c}: {n}  <-- NEW")

# Sanity check: 1020 + 1120 should equal the original count of 20s
n_20   = (events_raw[:, 2] == 20).sum()
n_1020 = (events_derived[:, 2] == 1020).sum()
n_1120 = (events_derived[:, 2] == 1120).sum()
print(f"\n=== Sanity check ===")
print(f"  Original 20s:                {n_20}")
print(f"  New 1020s (spatial array):   {n_1020}")
print(f"  New 1120s (symbolic array):  {n_1120}")
print(f"  1020 + 1120 =                {n_1020 + n_1120}")
print(f"  Match: {'OK' if n_1020 + n_1120 == n_20 else 'MISMATCH'}")

Raw events:     1680
Derived events: 1920
Difference:     +240 new events

=== Derived event codes (only new ones) ===
  1020: 120  <-- NEW
  1120: 120  <-- NEW

=== Sanity check ===
  Original 20s:                240
  New 1020s (spatial array):   120
  New 1120s (symbolic array):  120
  1020 + 1120 =                240
  Match: OK


In [11]:
from eeg_toolkit import derive_events_all

# Derive events for all included subjects.
# subj3 will be skipped (overwrite=False).
summary = derive_events_all(cfg, overwrite=False, verbose=True)

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj2', 'subj20', 'subj28', 'subj42', 'subj7']
Deriving events for 30 subject(s)

--- [1/30] subj3 ---
[subj3] derived events already exist — skipping (use overwrite=True to redo)

--- [2/30] subj4 ---
[subj4] deriving events
   raw events: 1680 total, 32 unique codes
   rule 'tag_memory_array_by_cue_type':
      20 -> 1020: 120 event(s)
      20 -> 1120: 120 event(s)
   derived events: 1920 total (+240 vs raw)
   saved: subj4_derived_events_eve.fif
   saved: subj4_event_recoding_log.txt

--- [3/30] subj5 ---
[subj5] deriving events
   raw events: 1680 total, 32 unique codes
   rule 'tag_memory_array_by_cue_type':
      20 -> 1020: 120 event(s)
      20 -> 1120: 120 event(s)
   derived events: 1920 total (+240 vs raw)
   saved: subj5_derived_events_eve.fif
   saved: subj5_event_recoding_log.txt

--- [4/30] subj6 ---
[subj6] deriving events
   raw events: 1680 total, 31 unique codes
   rule 'tag_memory_array_by_cue_t